# McCulloch-Pitts Neuron Model — Implementation
# McCulloch-Pitts 뉴런 모델 — 구현

**Paper**: McCulloch, W.S. & Pitts, W. (1943). "A Logical Calculus of the Ideas Immanent in Nervous Activity."

This notebook implements the McCulloch-Pitts (M-P) neuron model from scratch and demonstrates its key properties.

이 노트북은 McCulloch-Pitts (M-P) 뉴런 모델을 처음부터 구현하고 핵심 특성을 시연합니다.

## Contents / 목차
1. **Part 1**: M-P Neuron from Scratch / M-P 뉴런 직접 구현
2. **Part 2**: Logic Gates with M-P Neurons / M-P 뉴런으로 논리 게이트 구현
3. **Part 3**: Compound Gates (NAND, NOR, XOR) / 복합 게이트
4. **Part 4**: Heat-Cold Illusion Network / 열-냉 착각 네트워크 (논문 Fig. 1e)
5. **Part 5**: Feed-Forward Network Solver / 순방향 네트워크 시뮬레이션
6. **Part 6**: Feedback Loops (Circles) and Memory / 피드백 루프와 기억
7. **Part 7**: Linear Separability Visualization / 선형 분리 가능성 시각화
8. **Part 8**: Connection to Modern Neurons / 현대 뉴런과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: M-P Neuron from Scratch / M-P 뉴런 직접 구현

The McCulloch-Pitts neuron follows these rules:
- Collects binary inputs $x_1, x_2, \ldots, x_n$ from the previous time step
- Each input is either **excitatory** (weight +1) or **inhibitory** (absolute veto)
- If **any** inhibitory input is active → output = 0 (absolute inhibition)
- Otherwise, if the sum of excitatory inputs $\geq \theta$ (threshold) → output = 1
- All outputs have a **one time-step delay** (synaptic delay)

$$y(t) = \begin{cases} 1 & \text{if } \sum_{i \in \text{exc}} x_i(t-1) \geq \theta \text{ and } \forall j \in \text{inh}: x_j(t-1) = 0 \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
class MPNeuron:
    """McCulloch-Pitts neuron model (1943).

    Implements a binary threshold unit with absolute inhibition.
    All inputs are binary (0 or 1). Excitatory inputs contribute +1
    toward the threshold; any active inhibitory input vetoes firing.

    Attributes:
        threshold: Minimum excitatory sum required to fire.
        n_excitatory: Number of excitatory inputs.
        n_inhibitory: Number of inhibitory inputs.
    """

    def __init__(self, n_excitatory: int, n_inhibitory: int = 0,
                 threshold: int = 1):
        """Initialize M-P neuron.

        Args:
            n_excitatory: Number of excitatory input connections.
            n_inhibitory: Number of inhibitory input connections.
            threshold: Firing threshold (must be > 0).
        """
        self.threshold = threshold
        self.n_excitatory = n_excitatory
        self.n_inhibitory = n_inhibitory

    def fire(self, excitatory_inputs: list[int],
             inhibitory_inputs: list[int] | None = None) -> int:
        """Compute neuron output for given inputs.

        Args:
            excitatory_inputs: List of binary excitatory inputs.
            inhibitory_inputs: List of binary inhibitory inputs.

        Returns:
            1 if neuron fires, 0 otherwise.
        """
        if inhibitory_inputs is None:
            inhibitory_inputs = []

        # Absolute inhibition: any active inhibitory input blocks firing
        if any(i == 1 for i in inhibitory_inputs):
            return 0

        # Sum excitatory inputs and compare to threshold
        excitatory_sum = sum(excitatory_inputs)
        return 1 if excitatory_sum >= self.threshold else 0

    def truth_table(self) -> None:
        """Print the complete truth table for this neuron."""
        n_total = self.n_excitatory + self.n_inhibitory
        header_exc = [f"E{i+1}" for i in range(self.n_excitatory)]
        header_inh = [f"I{i+1}" for i in range(self.n_inhibitory)]
        header = header_exc + header_inh + ["OUT"]

        print(" | ".join(f"{h:>3}" for h in header))
        print("-" * (6 * len(header)))

        for inputs in product([0, 1], repeat=n_total):
            exc = list(inputs[:self.n_excitatory])
            inh = list(inputs[self.n_excitatory:])
            out = self.fire(exc, inh)
            row = list(inputs) + [out]
            print(" | ".join(f"{v:>3}" for v in row))


# Demonstrate basic neuron
print("=== M-P Neuron: threshold=2, 2 excitatory, 0 inhibitory ===")
print("(This is an AND gate)\n")
neuron = MPNeuron(n_excitatory=2, threshold=2)
neuron.truth_table()

## Part 2: Logic Gates with M-P Neurons / M-P 뉴런으로 논리 게이트 구현

Corresponding to Figure 1 (a)–(d) in the paper. Each basic logic gate is implemented by a single M-P neuron with appropriate threshold and inhibitory connections.

논문의 Figure 1 (a)–(d)에 대응합니다. 각 기본 논리 게이트는 적절한 threshold와 억제 연결을 가진 단일 M-P 뉴런으로 구현됩니다.

In [ ]:
def mp_and(a: int, b: int) -> int:
    """AND gate: fires only when both inputs are 1. (Fig. 1c)"""
    neuron = MPNeuron(n_excitatory=2, threshold=2)
    return neuron.fire([a, b])


def mp_or(a: int, b: int) -> int:
    """OR gate: fires when at least one input is 1. (Fig. 1b)"""
    neuron = MPNeuron(n_excitatory=2, threshold=1)
    return neuron.fire([a, b])


def mp_not(a: int) -> int:
    """NOT gate: fires when input is 0. Uses inhibitory input + always-on."""
    neuron = MPNeuron(n_excitatory=1, n_inhibitory=1, threshold=1)
    return neuron.fire([1], [a])  # always-on excitatory, a is inhibitory


def mp_and_not(a: int, b: int) -> int:
    """AND-NOT gate: fires when a=1 and b=0. (Fig. 1d)"""
    neuron = MPNeuron(n_excitatory=1, n_inhibitory=1, threshold=1)
    return neuron.fire([a], [b])


# Verify all gates against expected truth tables
gates = {
    "AND": (mp_and, lambda a, b: a & b),
    "OR": (mp_or, lambda a, b: a | b),
    "NOT": (mp_not, lambda a: 1 - a),
    "AND-NOT (a ∧ ¬b)": (mp_and_not, lambda a, b: a & (1 - b)),
}

print("Verifying M-P logic gates against expected truth tables:\n")
all_correct = True
for name, (mp_func, expected_func) in gates.items():
    if name == "NOT":
        results = [(a, mp_func(a), expected_func(a)) for a in [0, 1]]
        correct = all(r[1] == r[2] for r in results)
        status = "✓ PASS" if correct else "✗ FAIL"
        print(f"  {name}: {status}")
        for a, got, exp in results:
            print(f"    NOT({a}) = {got} (expected {exp})")
    else:
        results = [(a, b, mp_func(a, b), expected_func(a, b))
                    for a, b in product([0, 1], repeat=2)]
        correct = all(r[2] == r[3] for r in results)
        status = "✓ PASS" if correct else "✗ FAIL"
        print(f"  {name}: {status}")
        for a, b, got, exp in results:
            print(f"    {name}({a},{b}) = {got} (expected {exp})")
    all_correct &= correct
    print()

print("All gates correct!" if all_correct else "Some gates FAILED!")

## Part 3: Compound Gates (NAND, NOR, XOR) / 복합 게이트

These require **multi-layer networks** — composing basic M-P gates. XOR in particular requires at least 2 layers, which is the key limitation that Minsky & Papert (1969) would later formalize.

이것들은 **다층 네트워크**가 필요합니다. 특히 XOR는 최소 2개 층이 필요하며, 이것이 Minsky & Papert (1969)가 후에 형식화할 핵심 한계입니다.

In [ ]:
def mp_nand(a: int, b: int) -> int:
    """NAND = NOT(AND): 2-layer network."""
    return mp_not(mp_and(a, b))


def mp_nor(a: int, b: int) -> int:
    """NOR: fires only when both inputs are 0. Single neuron with 2 inhibitory."""
    neuron = MPNeuron(n_excitatory=1, n_inhibitory=2, threshold=1)
    return neuron.fire([1], [a, b])  # always-on excitatory, a & b inhibitory


def mp_xor(a: int, b: int) -> int:
    """XOR = (A OR B) AND NOT(A AND B): requires multi-layer network."""
    or_result = mp_or(a, b)
    and_result = mp_and(a, b)
    return mp_and_not(or_result, and_result)


# Verify compound gates
compound_gates = {
    "NAND": (mp_nand, lambda a, b: 1 - (a & b)),
    "NOR": (mp_nor, lambda a, b: 1 - (a | b)),
    "XOR": (mp_xor, lambda a, b: a ^ b),
}

print("Compound gates built from basic M-P neurons:\n")
for name, (mp_func, expected_func) in compound_gates.items():
    results = [(a, b, mp_func(a, b), expected_func(a, b))
                for a, b in product([0, 1], repeat=2)]
    correct = all(r[2] == r[3] for r in results)
    status = "✓ PASS" if correct else "✗ FAIL"
    print(f"  {name}: {status}")
    for a, b, got, exp in results:
        print(f"    {name}({a},{b}) = {got}")
    print()

## Part 4: Heat-Cold Illusion Network / 열-냉 착각 네트워크

The paper's practical example (p. 106, Fig. 1e): briefly touching a cold object produces a heat sensation. The network models this with temporal logic:

$$N_3(t) \equiv N_1(t-1) \vee [N_2(t-3) \cdot \sim N_2(t-2)]$$
$$N_4(t) \equiv N_2(t-2) \cdot N_2(t-1)$$

Where $N_1$ = heat receptor, $N_2$ = cold receptor, $N_3$ = heat sensation, $N_4$ = cold sensation.

논문의 실용 예시 (p. 106, Fig. 1e): 차가운 물체를 잠깐 대면 열감이 느껴지는 착각을 시뮬레이션합니다.

In [ ]:
def simulate_heat_cold(heat_input: list[int], cold_input: list[int],
                       n_steps: int) -> dict:
    """Simulate the heat-cold illusion network from Fig. 1e.

    Args:
        heat_input: Binary heat receptor values over time.
        cold_input: Binary cold receptor values over time.
        n_steps: Number of time steps to simulate.

    Returns:
        Dictionary with time series for all neurons.
    """
    n1 = list(heat_input) + [0] * (n_steps - len(heat_input))  # heat receptor
    n2 = list(cold_input) + [0] * (n_steps - len(cold_input))  # cold receptor
    n3 = [0] * n_steps  # heat sensation
    n4 = [0] * n_steps  # cold sensation

    for t in range(n_steps):
        # N3(t) = N1(t-1) OR [N2(t-3) AND NOT N2(t-2)]
        n1_prev = n1[t - 1] if t >= 1 else 0
        n2_prev2 = n2[t - 2] if t >= 2 else 0
        n2_prev3 = n2[t - 3] if t >= 3 else 0
        n3[t] = mp_or(n1_prev, mp_and_not(n2_prev3, n2_prev2))

        # N4(t) = N2(t-2) AND N2(t-1)
        n2_prev1 = n2[t - 1] if t >= 1 else 0
        n4[t] = mp_and(n2_prev2, n2_prev1)

    return {"N1 (heat receptor)": n1, "N2 (cold receptor)": n2,
            "N3 (heat sensation)": n3, "N4 (cold sensation)": n4}


# Scenario 1: Brief cold touch (1 time step)
print("Scenario 1: Brief cold touch at t=1")
result1 = simulate_heat_cold(
    heat_input=[0, 0, 0, 0, 0, 0, 0, 0],
    cold_input=[0, 1, 0, 0, 0, 0, 0, 0],  # cold for 1 step
    n_steps=8,
)

# Scenario 2: Prolonged cold touch (5 time steps)
print("Scenario 2: Prolonged cold touch at t=1..5")
result2 = simulate_heat_cold(
    heat_input=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    cold_input=[0, 1, 1, 1, 1, 1, 0, 0, 0, 0],  # cold for 5 steps
    n_steps=10,
)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

for ax, result, title in zip(axes, [result1, result2],
                              ["Brief cold touch (1 step)",
                               "Prolonged cold touch (5 steps)"]):
    n_steps = len(result["N1 (heat receptor)"])
    t = np.arange(n_steps)
    offsets = [3, 2, 1, 0]
    labels = list(result.keys())
    colors = ["#e74c3c", "#3498db", "#e67e22", "#2980b9"]

    for i, (label, offset, color) in enumerate(
            zip(labels, offsets, colors)):
        values = np.array(result[label])
        ax.fill_between(t, offset, offset + values * 0.8,
                         alpha=0.7, color=color, step="mid")
        ax.text(-0.5, offset + 0.4, label, ha="right", va="center",
                fontsize=9)

    ax.set_xlim(-0.5, n_steps - 0.5)
    ax.set_ylim(-0.3, 4.2)
    ax.set_xlabel("Time step (t)")
    ax.set_yticks([])
    ax.set_title(title)
    ax.set_xticks(t)

plt.tight_layout()
plt.suptitle("Heat-Cold Illusion (Fig. 1e)", y=1.02, fontsize=14)
plt.show()

print("\nKey observation:")
print("  Brief cold → heat sensation appears (illusion!)")
print("  Prolonged cold → only cold sensation (no heat illusion)")

## Part 5: Feed-Forward Network Solver / 순방향 네트워크 시뮬레이션

A general-purpose simulator for arbitrary M-P networks without circles (order 0). Each neuron is defined by its excitatory sources, inhibitory sources, and threshold.

원이 없는 임의의 M-P 네트워크를 위한 범용 시뮬레이터입니다. 각 뉴런은 흥분성 소스, 억제성 소스, threshold로 정의됩니다.

In [ ]:
class MPNetwork:
    """General-purpose McCulloch-Pitts network simulator.

    Supports arbitrary feed-forward and recurrent (circle) networks.
    Each neuron is defined by its excitatory sources, inhibitory sources,
    and threshold.
    """

    def __init__(self):
        """Initialize empty network."""
        self.neurons = {}  # name -> {exc_sources, inh_sources, threshold}
        self.state = {}    # name -> current value
        self.history = {}  # name -> list of values over time

    def add_neuron(self, name: str, exc_sources: list[str],
                   inh_sources: list[str] | None = None,
                   threshold: int = 1) -> None:
        """Add a neuron to the network.

        Args:
            name: Unique identifier for the neuron.
            exc_sources: Names of neurons providing excitatory input.
            inh_sources: Names of neurons providing inhibitory input.
            threshold: Firing threshold.
        """
        self.neurons[name] = {
            "exc_sources": exc_sources,
            "inh_sources": inh_sources or [],
            "threshold": threshold,
        }
        self.state[name] = 0
        self.history[name] = []

    def set_input(self, name: str, value: int) -> None:
        """Set the value of an input (peripheral afferent) neuron."""
        self.state[name] = value

    def step(self) -> dict[str, int]:
        """Advance the network by one time step.

        Returns:
            Dictionary of neuron states after the step.
        """
        # Save current state (inputs from t-1)
        prev_state = dict(self.state)

        # Record history
        for name in self.state:
            self.history[name].append(prev_state[name])

        # Compute new state for each non-input neuron
        for name, config in self.neurons.items():
            exc_vals = [prev_state.get(s, 0) for s in config["exc_sources"]]
            inh_vals = [prev_state.get(s, 0) for s in config["inh_sources"]]
            neuron = MPNeuron(len(exc_vals), len(inh_vals), config["threshold"])
            self.state[name] = neuron.fire(exc_vals, inh_vals)

        return dict(self.state)

    def simulate(self, input_sequences: dict[str, list[int]],
                 n_steps: int) -> dict[str, list[int]]:
        """Run the network for multiple time steps.

        Args:
            input_sequences: Dict mapping input neuron names to their
                time series of values.
            n_steps: Number of steps to simulate.

        Returns:
            History of all neuron states.
        """
        # Reset
        for name in self.state:
            self.state[name] = 0
            self.history[name] = []

        for t in range(n_steps):
            # Set inputs for this time step
            for name, seq in input_sequences.items():
                self.state[name] = seq[t] if t < len(seq) else 0
            self.step()

        return dict(self.history)


# Example: Build a 2-bit adder (half adder) using M-P neurons
# Sum = A XOR B, Carry = A AND B
net = MPNetwork()

# Input neurons (peripheral afferents)
net.state["A"] = 0
net.state["B"] = 0

# OR neuron: threshold=1, excitatory from A and B
net.add_neuron("OR", exc_sources=["A", "B"], threshold=1)

# AND neuron: threshold=2, excitatory from A and B
net.add_neuron("AND", exc_sources=["A", "B"], threshold=2)

# AND-NOT neuron: excitatory from OR, inhibitory from AND
net.add_neuron("XOR", exc_sources=["OR"], inh_sources=["AND"], threshold=1)

print("Half Adder (Sum = XOR, Carry = AND):")
print(f"{'A':>3} | {'B':>3} | {'Sum(XOR)':>8} | {'Carry(AND)':>10}")
print("-" * 35)

for a, b in product([0, 1], repeat=2):
    # Need 2 steps: inputs → OR/AND → XOR
    result = net.simulate(
        {"A": [a, a], "B": [b, b]},
        n_steps=2,
    )
    xor_val = result["XOR"][-1]
    and_val = result["AND"][-1]
    print(f"{a:>3} | {b:>3} | {xor_val:>8} | {and_val:>10}")

## Part 6: Feedback Loops (Circles) and Memory / 피드백 루프와 기억

Theorem 7 shows that feedback loops can implement memory — a neuron that, once activated, stays active indefinitely through self-excitation. This is the simplest "circle" in the paper.

정리 7은 피드백 루프가 기억을 구현할 수 있음을 보여줍니다 — 한 번 활성화되면 자기 흥분을 통해 무한정 활성 상태를 유지하는 뉴런입니다. 이것이 논문에서 가장 단순한 "원"입니다.

In [ ]:
def simulate_memory_circuit(trigger: list[int], reset: list[int],
                            n_steps: int) -> list[int]:
    """Simulate a 1-bit memory (SR latch) using M-P neurons with a circle.

    The memory neuron M excites itself (circle). Once triggered, it stays
    active until an inhibitory reset signal arrives.

    M(t) = [M(t-1) OR trigger(t-1)] AND NOT reset(t-1)

    Args:
        trigger: Binary set signal over time.
        reset: Binary reset signal over time.
        n_steps: Number of simulation steps.

    Returns:
        Memory neuron state over time.
    """
    m = [0] * n_steps
    for t in range(1, n_steps):
        trig = trigger[t - 1] if t - 1 < len(trigger) else 0
        rst = reset[t - 1] if t - 1 < len(reset) else 0
        # Self-excitation (circle) OR trigger, vetoed by reset
        m_or_trig = mp_or(m[t - 1], trig)
        m[t] = mp_and_not(m_or_trig, rst)
    return m


n_steps = 20
trigger = [0] * 20
trigger[3] = 1   # Set at t=3

reset = [0] * 20
reset[10] = 1    # Reset at t=10

trigger2 = [0] * 20
trigger2[14] = 1  # Set again at t=14

# Combine triggers
combined_trigger = [t1 | t2 for t1, t2 in zip(trigger, trigger2)]

memory = simulate_memory_circuit(combined_trigger, reset, n_steps)

fig, ax = plt.subplots(figsize=(12, 4))
t = np.arange(n_steps)

ax.step(t, np.array(combined_trigger) * 0.8 + 2.2, where="mid",
        color="#2ecc71", linewidth=2, label="Trigger (set)")
ax.step(t, np.array(reset) * 0.8 + 1.1, where="mid",
        color="#e74c3c", linewidth=2, label="Reset")
ax.step(t, np.array(memory) * 0.8, where="mid",
        color="#3498db", linewidth=2.5, label="Memory (M)")

ax.fill_between(t, 0, np.array(memory) * 0.8,
                 alpha=0.3, color="#3498db", step="mid")

ax.set_xlabel("Time step (t)")
ax.set_yticks([0.4, 1.5, 2.6])
ax.set_yticklabels(["Memory", "Reset", "Trigger"])
ax.set_title("1-Bit Memory via Feedback Loop (Circle)")
ax.set_xticks(t)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("The circle (self-excitation) allows the neuron to 'remember'")
print("its state indefinitely — implementing Theorem 7's key insight.")

## Part 7: Linear Separability Visualization / 선형 분리 가능성 시각화

A single M-P neuron can only implement **linearly separable** functions — those where a straight line (or hyperplane) can separate the 1-outputs from the 0-outputs. AND and OR are linearly separable; XOR is not.

단일 M-P 뉴런은 **선형 분리 가능**한 함수만 구현할 수 있습니다 — 직선(또는 초평면)으로 출력 1과 출력 0을 분리할 수 있는 함수입니다. AND와 OR는 선형 분리 가능하지만, XOR는 불가능합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

gate_configs = [
    ("AND", mp_and, {"x": 1.5, "slope": -1, "intercept": 1.5}),
    ("OR", mp_or, {"x": 0.5, "slope": -1, "intercept": 0.5}),
    ("XOR", mp_xor, None),  # No single line can separate
]

for ax, (name, func, line_params) in zip(axes, gate_configs):
    inputs = list(product([0, 1], repeat=2))
    for a, b in inputs:
        out = func(a, b)
        color = "#2ecc71" if out == 1 else "#e74c3c"
        marker = "o" if out == 1 else "x"
        ax.scatter(a, b, c=color, s=200, marker=marker, zorder=5,
                    edgecolors="black", linewidths=1.5)

    # Draw decision boundary if linearly separable
    if line_params is not None:
        x_line = np.linspace(-0.3, 1.3, 100)
        y_line = line_params["slope"] * x_line + line_params["intercept"]
        ax.plot(x_line, y_line, "k--", linewidth=1.5, alpha=0.7)
        ax.fill_between(x_line, y_line, 1.5, alpha=0.1, color="#2ecc71")
    else:
        ax.text(0.5, -0.25, "No single line can separate!",
                ha="center", fontsize=10, color="red", fontweight="bold")

    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.4, 1.4)
    ax.set_xlabel("Input A")
    ax.set_ylabel("Input B")
    ax.set_title(f"{name} Gate")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ecc71",
            markersize=12, label="Output = 1"),
    Line2D([0], [0], marker="x", color="#e74c3c", markersize=12,
            markeredgewidth=2, label="Output = 0", linestyle="None"),
    Line2D([0], [0], color="black", linestyle="--", label="Decision boundary"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle("Linear Separability: Why XOR Needs Multi-Layer Networks",
             fontsize=13)
plt.tight_layout()
plt.show()

## Part 8: Connection to Modern Neurons / 현대 뉴런과의 연결

The M-P neuron uses a **step function** (all-or-none). Modern neural networks replace this with smooth, differentiable activation functions that enable gradient-based learning.

M-P 뉴런은 **계단 함수**(all-or-none)를 사용합니다. 현대 신경망은 이를 미분 가능한 부드러운 활성화 함수로 대체하여 gradient 기반 학습을 가능하게 합니다.

In [ ]:
x = np.linspace(-6, 6, 300)

# Activation functions
step = np.where(x >= 0, 1, 0)  # McCulloch-Pitts (1943)
sigmoid = 1 / (1 + np.exp(-x))  # Logistic sigmoid
tanh = np.tanh(x)               # Hyperbolic tangent
relu = np.maximum(0, x)          # ReLU (modern default)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

configs = [
    ("Step (M-P 1943)", step, "#e74c3c",
     "Binary: 0 or 1\nNo gradient\nAll-or-none"),
    ("Sigmoid", sigmoid, "#3498db",
     "Smooth: (0, 1)\nVanishing gradient\nLogistic curve"),
    ("Tanh", tanh, "#2ecc71",
     "Smooth: (-1, 1)\nZero-centered\nBetter than sigmoid"),
    ("ReLU (Modern)", relu, "#9b59b6",
     "Piecewise linear\nNo vanishing gradient\nDefault today"),
]

for ax, (name, values, color, desc) in zip(axes, configs):
    ax.plot(x, values, color=color, linewidth=2.5)
    ax.axhline(y=0, color="gray", linewidth=0.5)
    ax.axvline(x=0, color="gray", linewidth=0.5)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlim(-6, 6)
    ax.text(0.05, 0.95, desc, transform=ax.transAxes,
            fontsize=8, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    ax.grid(True, alpha=0.2)

plt.suptitle("Evolution of Activation Functions: M-P (1943) → Modern",
             fontsize=13)
plt.tight_layout()
plt.show()

print("Key evolution from McCulloch-Pitts to modern neurons:")
print("  1943: Binary step function → No learning possible")
print("  1958: Perceptron learning rule (Rosenblatt) → Linear models")
print("  1986: Backpropagation + sigmoid (Rumelhart) → Multi-layer learning")
print("  2012: ReLU + deep networks (Krizhevsky) → Modern deep learning")

## Summary / 요약

| Part | What We Built / 구현 내용 | Key Insight / 핵심 통찰 |
|------|------------------------|---------------------|
| 1 | M-P neuron class | Binary threshold + absolute inhibition = simplest possible neuron |
| 2 | AND, OR, NOT, AND-NOT gates | Single neurons compute basic logic (Theorem 2) |
| 3 | NAND, NOR, XOR | Compound gates need multi-layer networks |
| 4 | Heat-cold illusion | Temporal logic creates perceptual illusions |
| 5 | General network simulator | Any TPE is realizable (Theorem 2) |
| 6 | Memory via feedback loop | Circles enable memory (Theorem 7) |
| 7 | Linear separability plot | XOR is not linearly separable → single neuron limit |
| 8 | Activation function evolution | Step → sigmoid → ReLU: the path to modern deep learning |

**Next paper**: Turing (1950) — "Computing Machinery and Intelligence" — asks whether machines can *think*.

**다음 논문**: Turing (1950) — "Computing Machinery and Intelligence" — 기계가 *생각*할 수 있는지 묻습니다.